In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.optim import Adam

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

# 1) Convert to Tensors
# Images: float32, Labels: float32
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).view(-1, 1)



In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)


In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  drop_last=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, drop_last=False)


In [ ]:
# 4. Print shape of one batch

xb, yb = next(iter(train_loader))
print("Batch X shape:", xb.shape)
print("Batch y shape:", yb.shape)


In [ ]:
# 5. Display sample images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
for i in range(8):
    img = xb[i].permute(1, 2, 0).numpy()  # H, W, C
    axes[i].imshow(img)
    axes[i].set_title(f"Age: {yb[i].item():.0f}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:
class AgeRegressor(nn.Module):
    def __init__(self, in_shape):
        super().__init__()
        c, h, w = in_shape
        in_features = c * h * w

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 512),  # 1st
            nn.ReLU(),
            nn.Linear(512, 256),          # 2nd
            nn.ReLU(),
            nn.Linear(256, 128),          # 3rd
            nn.ReLU(),
            nn.Linear(128, 1)             # (output)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        preds = model(x)
        loss = loss_fn(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

In [ ]:
# Task 3: Write your validation loop here:
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            preds = model(x)
            loss = loss_fn(preds, y)

            total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

In [ ]:
# Task 4: Define device, model, loss, optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

xb, yb = next(iter(train_loader))
model = AgeRegressor(in_shape=xb.shape[1:]).to(device)

loss_fn = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=1e-3)

In [ ]:
# Task 5: Start training for 20 epochs:

num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss = validate_one_epoch(model, test_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch:02d}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
# Task 1: Write your code here:

import matplotlib.pyplot as plt
import torch

# 1) Plot training & validation
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval()

xb, yb = next(iter(test_loader))
xb = xb.to(device)

with torch.no_grad():
    preds = model(xb).cpu().squeeze(1)
actual = yb.squeeze(1)

# first 8 images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i in range(8):
    img = xb[i].cpu().permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f"Pred: {preds[i].item():.1f} | True: {actual[i].item():.1f}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()